In [1]:
2+2


4

In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.tools  import WikipediaQueryRun
from dotenv import load_dotenv
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()

True

In [3]:
llm =ChatOpenAI(model="gpt-4o-mini",temperature=0.5)


In [4]:
wiki_wrapper=WikipediaAPIWrapper(top_k_results=3,doc_content_chars_max=400)
wiki_tool=WikipediaQueryRun(api_wrapper=wiki_wrapper,description="query wikipedia")



In [5]:
wiki_tool.name

'wikipedia'

In [6]:
wiki_tool.invoke("what is genai")


'Page: Generative artificial intelligence\nSummary: Generative artificial intelligence (Generative AI or GenAI) is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data and use them to produce new data in response to input, which '

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool=TavilySearchResults()

result=tavily_tool.invoke({"query": "what is genai"})
print(type(result))
r = result[0]
print(r["title"])
print(r["url"])
print(r["content"])

/var/folders/tc/f5d05_ts38v4p_s_j0zrgq1m0000gn/T/ipykernel_30723/3678674916.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool=TavilySearchResults()


<class 'list'>
Generative artificial intelligence - Wikipedia
https://en.wikipedia.org/wiki/Generative_artificial_intelligence
Generative artificial intelligence (Generative AI or GenAI) is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data and use them to produce new data( in response to input, which often comes in the form of natural language prompts "Prompt (natural language)").(


In [8]:
def add_tool(a:int,b:int)->int:
    """This tool accepts two paramerters of type int for addition
    and returns the output as sum of two numbers in int"""
    return a+b

In [9]:
tools=[tavily_tool,add_tool]
llm_with_tools=llm.bind_tools(tools=tools)

llm_with_tools.invoke("what is genAI")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 127, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CvhHX63eGtIqd5ShL8m4dbCVeQ6KJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9d17-47ca-7e82-8497-0af0a6c365da-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'what is genAI'}, 'id': 'call_zmPSTqvyaZXKFCpWJnf4G1QO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 21, 'total_tokens': 148, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

In [10]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict,Annotated
from langgraph.checkpoint.memory import InMemorySaver



In [23]:
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

def llm_calling(state:State):
    return {
        "messages":[llm_with_tools.invoke(state["messages"])]
    }


In [25]:
graph=StateGraph(State)

graph.add_node("llm_calling",llm_calling)
graph.add_node("tools",ToolNode(tools=tools,messages_key="messages"))

graph.add_edge(START,"llm_calling")
graph.add_conditional_edges("llm_calling",tools_condition)
graph.add_edge("tools","llm_calling")
graph.add_edge("llm_calling",END)

memory=InMemorySaver()
workflow=graph.compile(checkpointer=memory)
config={"configurable":{"thread_id":2}}

print(workflow.get_graph().draw_mermaid())



---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_calling(llm_calling)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_calling;
	llm_calling -.-> __end__;
	llm_calling -.-> tools;
	tools --> llm_calling;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [26]:
user_input = "What is current temperature in kannauj,uttar Pradesh"
events = workflow.stream(
    {"messages": [("user", user_input)]},config=config, stream_mode="values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

What is current temperature in kannauj,uttar Pradesh
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (call_oPyl7I2C3L0Peha8sGmIjfsY)
 Call ID: call_oPyl7I2C3L0Peha8sGmIjfsY
  Args:
    query: current temperature in Kannauj, Uttar Pradesh
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "Kannauj, Uttar Pradesh, India 14 day weather forecast - Time and Date", "url": "https://www.timeanddate.com/weather/india/kannauj/ext", "content": "| Sun Jan 18 |  | 71 / 44 °F | Sunny. | 76 °F | 8 mph | ↑ | 30% | 1% | 0.00\" | 3 (Moderate) | 7:00 am | 5:40 pm |\n| Mon Jan 19 |  | 71 / 44 °F | Sunny. | 76 °F | 12 mph | ↑ | 31% | 1% | 0.00\" | 3 (Moderate) | 7:00 am | 5:41 pm |\n| Tue Jan 20 |  | 71 / 44 °F | Sunny. | 76 °F | 12 mph | ↑ | 30% | 1% | 0.00\" | 3 (Moderate) | 7:

In [27]:
user_input = "What i asked previously"
events = workflow.stream(
    {"messages": [("user", user_input)]},config=config, stream_mode="values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

What i asked previously
================================== Ai Message ==================================

You asked for the current temperature in Kannauj, Uttar Pradesh. The response indicated that the temperature is approximately 56°F (about 13°C), with the weather described as decreasing clouds and cool. If you need more specific information or updates, please let me know!


In [20]:
response=workflow.invoke({"messages":"what is GENAI"},config=config,stream_mode="values")

for r in response["messages"]:
     r.pretty_print()

================================ Human Message =================================

What is current temperature in kannauj,uttar Pradesh
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (call_131WBSLIFBPRIxbsY23o4oBT)
 Call ID: call_131WBSLIFBPRIxbsY23o4oBT
  Args:
    query: current temperature in Kannauj, Uttar Pradesh
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "10-Day Weather Forecast - Kannauj, Uttar Pradesh, IN", "url": "https://www.weatherbug.com/weather-forecast/10-day-weather/kannauj-uttar-pradesh-in", "content": "# 10-Day Weather Forecast - Kannauj, Uttar Pradesh, IN\n\n## Thu, Jan 8\n\n### Today\n\nMostly cloudy. High temperature around 64F. Dew point will be around 44F with an average humidity of 35%. Winds will be 3 mph from the N.\n\n### Tonight\n\nPartly cloudy. Low temperature around 45F. Dew point will be around 43F wi